# Day 2 – Experimental Design and A/B Testing
## Student Practice Notebook

**Programme:** Educentriks – Applied Data Science, AI/ML & Generative AI  
**Session duration:** 90 minutes  
**Business scenario:** E-commerce checkout experiment  
**Dataset:** Synthetic A/B Testing (`synthetic_ab_testing.csv`)

---

### How to use this notebook

1. Read each task before running the code.
2. Run the completed setup and data-generation cells.
3. Complete the sections marked **YOUR TASK**.
4. Use the hints if you get stuck, but try the task before opening the hint.
5. Write business interpretations in your own words.

This is a practice notebook. Some analysis cells are intentionally incomplete. Your instructor has a separate completed notebook.


## 1. Learning objectives

By completing this notebook, you will practise how to:

- Understand control and treatment groups in an A/B test.
- Work with a reproducible synthetic experiment dataset.
- Calculate and compare conversion rates.
- Create a contingency table.
- Apply a chi-square test of independence.
- Explain what statistical evidence means for a business decision.

### Business problem

An e-commerce company has redesigned its checkout page. The current design is **Version A (control)** and the redesigned page is **Version B (treatment)**.

The company wants to determine whether the redesigned checkout is associated with a change in the proportion of visitors who complete a purchase.


## 2. Experiment design

Review the experiment setup before analysing the data.

| Element | Description |
|---|---|
| Control group | Version A – existing checkout |
| Treatment group | Version B – redesigned checkout |
| Visitors | 1,000 in each group |
| Outcome variable | `converted` |
| Conversion coding | 1 = converted; 0 = not converted |
| Significance level | 0.05 |

### Hypotheses

Complete the statements in your own words:

- **Null hypothesis (H₀):** There is ________________________________ between experiment group and conversion outcome.
- **Alternative hypothesis (H₁):** There is __________________________ between experiment group and conversion outcome.

Remember: a p-value is not the probability that the null hypothesis is true.


## 3. Environment setup

Required packages:

- `faker`
- `numpy`
- `pandas`
- `scipy`

If a package is missing, install it in your active `.venv`:

```bash
python -m pip install faker numpy pandas scipy
```

Select your course `.venv` as the Jupyter kernel in VS Code.


In [ ]:
# Import the libraries required for this exercise.
import numpy as np
import pandas as pd
from faker import Faker
from scipy.stats import chi2_contingency

# Make pandas output easier to read.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 4. Generate the synthetic experiment dataset

The dataset is generated locally. You do not need to download a CSV file.

The generation logic is supplied so that everyone starts with the same classroom dataset. The visitor attributes are synthetic. The conversion outcome is generated separately for the two experiment groups.

**Important:** The probabilities in the generator are simulation settings, not results from a real company or a real experiment.


In [ ]:
# Set up reproducible random generators.
fake = Faker()
fake.seed_instance(42)

rng = np.random.default_rng(42)
rng_a = np.random.default_rng(11)
rng_b = np.random.default_rng(83)

# Define the number of visitors and simulation probabilities.
N_PER_GROUP = 1000
CONTROL_CONVERSION_PROBABILITY = 0.10
TREATMENT_CONVERSION_PROBABILITY = 0.13

# Generate the binary conversion outcome for each group.
# 1 means converted; 0 means not converted.
group_a_converted = rng_a.binomial(
    n=1,
    p=CONTROL_CONVERSION_PROBABILITY,
    size=N_PER_GROUP
)

group_b_converted = rng_b.binomial(
    n=1,
    p=TREATMENT_CONVERSION_PROBABILITY,
    size=N_PER_GROUP
)

# Combine group labels and conversion outcomes.
experiment_group = np.array(
    ["A"] * N_PER_GROUP + ["B"] * N_PER_GROUP
)
converted = np.concatenate([group_a_converted, group_b_converted])

total_visitors = N_PER_GROUP * 2

# Generate synthetic device and traffic-source attributes.
device_type = rng.choice(
    ["Desktop", "Mobile", "Tablet"],
    size=total_visitors,
    p=[0.35, 0.55, 0.10]
)

traffic_source = rng.choice(
    ["Organic Search", "Paid Search", "Social Media", "Email", "Referral"],
    size=total_visitors,
    p=[0.30, 0.25, 0.18, 0.17, 0.10]
)

# Generate plausible synthetic session behaviour.
session_duration_sec = np.clip(
    rng.normal(loc=180, scale=65, size=total_visitors),
    20,
    600
).round().astype(int)

pages_viewed = np.clip(
    rng.poisson(lam=4, size=total_visitors) + 1,
    1,
    20
)

# Generate synthetic visit dates within the recent 30-day period.
visit_date = [
    fake.date_between(start_date="-30d", end_date="today")
    for _ in range(total_visitors)
]

# Assign order value only to visitors who converted.
order_value = np.where(
    converted == 1,
    np.round(rng.lognormal(mean=3.7, sigma=0.55, size=total_visitors), 2),
    0.0
)

# Create synthetic visitor IDs.
visitor_id = [
    f"VIS-{visitor_number:05d}"
    for visitor_number in range(1, total_visitors + 1)
]

# Assemble all fields into a pandas DataFrame.
ab_data = pd.DataFrame({
    "visitor_id": visitor_id,
    "visit_date": visit_date,
    "device_type": device_type,
    "traffic_source": traffic_source,
    "experiment_group": experiment_group,
    "converted": converted,
    "session_duration_sec": session_duration_sec,
    "pages_viewed": pages_viewed,
    "order_value": order_value
})

# Shuffle rows so the records are not grouped by experiment version.
ab_data = ab_data.iloc[rng.permutation(total_visitors)].reset_index(drop=True)

print("Dataset generated.")
print("Number of records:", len(ab_data))


## 5. Inspect the dataset

### Task 1 – Explore the data

Run the following cells and answer the questions in the Markdown cell below.

- Display the first five records.
- Check the number of rows and columns.
- Review the data types and missing values.
- Confirm that every visitor ID is unique.


In [ ]:
# YOUR TASK 1A:
# Display the first five records.
# Replace the placeholder with the correct pandas command.



In [ ]:
# YOUR TASK 1B:
# Display the number of rows and columns.



In [ ]:
# YOUR TASK 1C:
# Inspect column data types and non-null counts.



In [ ]:
# YOUR TASK 1D:
# Check the total number of missing values.



In [ ]:
# YOUR TASK 1E:
# Check whether visitor_id contains unique values.
# Hint: use nunique() and compare it with the number of rows.



### Record your observations

1. How many visitors are in the dataset?
2. How many columns are available?
3. Does every visitor have a unique ID?
4. Are there any missing values?
5. Which column identifies the control and treatment groups?


## 6. Save the dataset

### Task 2 – Export to CSV

Save the generated DataFrame as `synthetic_ab_testing.csv` in your current working directory.

**YOUR TASK:** Complete the export statement. Ensure that the DataFrame index is not written as an extra column.


In [ ]:
# YOUR TASK 2:
# Export ab_data to synthetic_ab_testing.csv.
# Hint: DataFrame.to_csv()



## 7. Calculate conversion rates

A conversion rate measures the proportion of visitors who completed the desired action.

\[
\text{Conversion Rate} = \frac{\text{Conversions}}{\text{Total Visitors}} \times 100
\]

### Task 3 – Group-level performance

Create a summary grouped by `experiment_group` with these measures:

- `visitors`: count of visitor records
- `conversions`: sum of the `converted` column
- `conversion_rate`: mean of the `converted` column

**YOUR TASK:** Complete the aggregation below. Use `groupby()` and `agg()`.


In [ ]:
# YOUR TASK 3:
conversion_summary = (
    ab_data
    .groupby("experiment_group")
    .agg(
        # visitors = count of visitor_id
        # conversions = sum of converted
        # conversion_rate = mean of converted
    )
    .reset_index()
)

conversion_summary


### Task 4 – Compare the groups

Using your summary:

1. Which group has the higher conversion rate?
2. What is the absolute difference in percentage points?
3. Calculate the relative uplift of B compared with A.

Use these definitions:

- **Absolute difference:** Treatment rate − Control rate.
- **Relative uplift:** (Treatment rate − Control rate) ÷ Control rate.

**YOUR TASK:** Complete the calculations. Format the rates as percentages when printing them.


In [ ]:
# YOUR TASK 4:
# Extract the conversion rate for group A and group B.
rate_a = None  # Replace None with your code.
rate_b = None  # Replace None with your code.

# Calculate absolute difference and relative uplift.
absolute_difference = None  # Replace None with your code.
relative_uplift = None       # Replace None with your code.

# Print the results in a readable format.



## 8. Create a contingency table

A contingency table displays counts for combinations of categorical variables.

For this experiment, compare:
- `experiment_group`: A or B
- `converted`: 0 or 1

### Task 5 – Build the table

Create a cross-tabulation with experiment groups as rows and conversion outcomes as columns.

**YOUR TASK:** Use `pd.crosstab()` and assign the result to `contingency_table`.


In [ ]:
# YOUR TASK 5:
contingency_table = None  # Replace None with your pd.crosstab() command.

contingency_table


### Interpret the table

1. What does the cell at row A and column 1 represent?
2. What does the cell at row B and column 0 represent?
3. Why should we examine both the counts and the conversion rates?


## 9. Run a chi-square test of independence

The chi-square test of independence helps assess whether two categorical variables are associated.

For this experiment:

- H₀: Experiment group and conversion outcome are independent.
- H₁: Experiment group and conversion outcome are associated.
- Significance level: α = 0.05.

### Decision rule

- If p-value is less than 0.05, reject H₀.
- If p-value is 0.05 or greater, do not reject H₀.

### Task 6 – Perform the test

Use `chi2_contingency()` with your contingency table.

Capture:
- Chi-square statistic
- p-value
- Degrees of freedom
- Expected frequencies

**YOUR TASK:** Complete the code.


In [ ]:
# YOUR TASK 6:
# Run chi2_contingency() using contingency_table.
# The function returns four values.

chi2_statistic = None
p_value = None
degrees_of_freedom = None
expected_frequencies = None

# Print the statistic, p-value and degrees of freedom.



In [ ]:
# YOUR TASK 6B:
# Compare the p-value with alpha = 0.05.
# Print either:
# "Reject the null hypothesis"
# or
# "Do not reject the null hypothesis"

alpha = 0.05



### Task 7 – Check expected frequencies

A key assumption of the chi-square test is that expected cell frequencies should be sufficiently large. A common classroom rule is that expected counts should not be below 5.

**YOUR TASK:** Display the expected frequencies returned by the test and identify the smallest expected cell count.

Do not confuse observed counts in the contingency table with expected counts calculated by the statistical test.


In [ ]:
# YOUR TASK 7:
# Convert expected_frequencies into a DataFrame using the same
# row and column labels as contingency_table.

expected_table = None  # Replace None with your code.

expected_table


## 10. Business interpretation

### Task 8 – Write a short business recommendation

Use your calculated results to write a short recommendation to the e-commerce product manager.

Your response must cover:

1. Which checkout version has the higher observed conversion rate?
2. What is the absolute difference in percentage points?
3. What does the p-value indicate at the 5% significance level?
4. What is one reason the business should not make a rollout decision based only on the p-value?
5. What additional business metric would you review before recommending a full rollout?

### Important interpretation reminders

- Statistical significance does not automatically mean business significance.
- A p-value is not the probability that H₀ is true.
- A sound experiment design is essential before making causal claims.
- Consider revenue, order value, costs, customer experience and operational risks.


## 11. Independent challenge

### Challenge – Examine conversion by device

The business wants to know whether the observed conversion pattern is similar across devices.

Complete the following:

1. Group the dataset by `device_type` and `experiment_group`.
2. Calculate visitor count, conversion count and conversion rate for each combination.
3. Identify which device/group combination has the highest observed conversion rate.
4. Write one caution about interpreting this segmented comparison.

This is an exploratory extension. Do not claim that a subgroup difference is statistically significant unless you conduct an appropriate analysis and account for the limitations of multiple comparisons and sample size.


In [ ]:
# YOUR TASK 9:
# Group by device_type and experiment_group.
# Calculate visitors, conversions and conversion_rate.

device_summary = None  # Replace None with your groupby/agg code.

device_summary


## 12. Knowledge check

Answer these questions without referring to the training notes.

1. What is the purpose of a control group in an A/B test?
2. Why is random assignment important?
3. What is the difference between a conversion count and a conversion rate?
4. What does the null hypothesis state in this experiment?
5. What does a p-value below 0.05 indicate under the test assumptions?
6. Does statistical significance automatically mean the treatment should be launched? Explain briefly.

### Submission checklist

Before submitting your notebook, confirm that:

- [ ] All setup and dataset-generation cells run successfully.
- [ ] You inspected the dataset and checked its quality.
- [ ] You exported `synthetic_ab_testing.csv`.
- [ ] You calculated group-level conversion rates.
- [ ] You created the contingency table.
- [ ] You ran the chi-square test.
- [ ] You checked expected frequencies.
- [ ] You wrote a business recommendation.
- [ ] You completed the device-level challenge.
- [ ] You answered the knowledge-check questions.

**Next session:** Day 3 – ML Pipeline and Bias–Variance.
